# My Summary: The Main Data Loading Pipeline

- I pulled out the main data loading pipeline from Chapter 2 so I can reuse it
- This notebook shows the minimal code: tokenizer → dataset → dataloader → embedding usage

The full step-by-step chapter code is in [01_ch02.ipynb](./01_ch02.ipynb).

My takeaway: this notebook has just the data loading pipeline so I can copy or reference it without the intermediate explanations.

Packages I'm using in this notebook:

In [ ]:
# NBVAL_SKIP
from importlib.metadata import version as pkg_version

print(f"Torch version in my environment: {pkg_version('torch')}")
print(f"tiktoken version in my environment: {pkg_version('tiktoken')}")

In [ ]:
import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset


class SlidingWindowTokenDataset(Dataset):
    def __init__(self, text: str, tokenizer, window_size: int, stride: int) -> None:
        encoded_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.examples = []

        for start_index in range(0, len(encoded_text) - window_size, stride):
            input_ids = torch.tensor(encoded_text[start_index : start_index + window_size])
            next_token_targets = torch.tensor(
                encoded_text[start_index + 1 : start_index + window_size + 1]
            )
            self.examples.append((input_ids, next_token_targets))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int):
        return self.examples[index]


def build_token_dataloader(
    text: str,
    batch_size: int,
    window_size: int,
    stride: int,
    shuffle: bool = True,
    drop_last: bool = True,
    num_workers: int = 0,
) -> DataLoader:
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = SlidingWindowTokenDataset(text, tokenizer, window_size=window_size, stride=stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )


with open("the-verdict.txt", "r", encoding="utf-8") as file_handle:
    book_text = file_handle.read()

vocab_size = 50257
embedding_dim = 256
context_length = 1024
batch_size = 8
window_size = 4

token_embedding = torch.nn.Embedding(vocab_size, embedding_dim)
position_embedding = torch.nn.Embedding(context_length, embedding_dim)

dataloader = build_token_dataloader(
    book_text,
    batch_size=batch_size,
    window_size=window_size,
    stride=window_size,
)

In [ ]:
for input_batch, target_batch in dataloader:
    learned_token_vectors = token_embedding(input_batch)
    learned_position_vectors = position_embedding(torch.arange(window_size))
    combined_embeddings = learned_token_vectors + learned_position_vectors
    break

In [ ]:
print(combined_embeddings.shape)

My takeaway: input/target pairs are just token sequences shifted by one; the dataloader feeds these in batches so I can train the model on next-token prediction.